In [16]:
"""
FRACTURE DETECTION – MSc CS PROJECT (FULL PIPELINE + POLISH)
"""

'\nFRACTURE DETECTION – MSc CS PROJECT (FULL PIPELINE + POLISH)\n'

In [17]:
# =========================
# IMPORTS
# =========================

import os
import cv2
import json
import numpy as np
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

print("TensorFlow:", tf.__version__)
print("GPU:", len(tf.config.list_physical_devices("GPU")) > 0)

TensorFlow: 2.18.0
GPU: True


In [18]:
# =========================
# CONFIG
# =========================
CONFIG = {
    "img_size": (224, 224),
    "batch_size": 16,     # reduced for memory safety
    "epochs": 120,
    "learning_rate": 3e-4,
    "test_size": 0.2,
    "seed": 42
}

np.random.seed(CONFIG["seed"])
tf.random.set_seed(CONFIG["seed"])

In [19]:
# =========================
# DATASET PATH (VERIFIED)
# =========================
DATA_DIR = "/kaggle/input/bone-fracture-dataset/Ultimate-Mixed-Bone-Fracture-Dataset 1/images"
assert os.path.exists(DATA_DIR), "❌ Dataset path invalid"

print("✅ Dataset found:", DATA_DIR)
print("Classes:", os.listdir(DATA_DIR))

✅ Dataset found: /kaggle/input/bone-fracture-dataset/Ultimate-Mixed-Bone-Fracture-Dataset 1/images
Classes: ['Non_fractured', 'Fractured']


In [20]:
# =========================
# EXACT DUPLICATE DETECTION (PIXEL-IDENTICAL)
# =========================

import hashlib
from tqdm import tqdm

def file_md5(path, chunk_size=8192):
    """Compute MD5 hash of a file (exact duplicate detection)."""
    md5 = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            md5.update(chunk)
    return md5.hexdigest()


print("\n🔍 Checking for exact duplicate images...")

hash_to_path = {}
duplicate_files = []

all_image_paths = []

for cls in os.listdir(DATA_DIR):
    cls_path = os.path.join(DATA_DIR, cls)
    if not os.path.isdir(cls_path):
        continue

    for f in os.listdir(cls_path):
        if f.lower().endswith(('.png','.jpg','.jpeg','.bmp','.tif','.tiff')):
            all_image_paths.append(os.path.join(cls_path, f))

print("Total images found:", len(all_image_paths))

for path in tqdm(all_image_paths):
    try:
        h = file_md5(path)
        if h in hash_to_path:
            duplicate_files.append((hash_to_path[h], path))
        else:
            hash_to_path[h] = path
    except:
        pass

print("\nExact duplicate pairs found:", len(duplicate_files))

# Show a few examples (optional)
if duplicate_files:
    print("\nSample duplicate pairs:")
    for a, b in duplicate_files[:5]:
        print(" -", a)
        print("   ", b)


🔍 Checking for exact duplicate images...
Total images found: 15456


100%|██████████| 15456/15456 [00:28<00:00, 539.21it/s]


Exact duplicate pairs found: 6557

Sample duplicate pairs:
 - /kaggle/input/bone-fracture-dataset/Ultimate-Mixed-Bone-Fracture-Dataset 1/images/Non_fractured/64-rotated2-rotated2-rotated1.jpg
    /kaggle/input/bone-fracture-dataset/Ultimate-Mixed-Bone-Fracture-Dataset 1/images/Non_fractured/64-rotated2-rotated2-rotated1 (1).jpg
 - /kaggle/input/bone-fracture-dataset/Ultimate-Mixed-Bone-Fracture-Dataset 1/images/Non_fractured/62-rotated3-rotated1 (1).jpg
    /kaggle/input/bone-fracture-dataset/Ultimate-Mixed-Bone-Fracture-Dataset 1/images/Non_fractured/62-rotated3-rotated1.jpg
 - /kaggle/input/bone-fracture-dataset/Ultimate-Mixed-Bone-Fracture-Dataset 1/images/Non_fractured/63-rotated3-rotated3-rotated1-rotated1 (1).jpg
    /kaggle/input/bone-fracture-dataset/Ultimate-Mixed-Bone-Fracture-Dataset 1/images/Non_fractured/63-rotated3-rotated3-rotated1-rotated1.jpg
 - /kaggle/input/bone-fracture-dataset/Ultimate-Mixed-Bone-Fracture-Dataset 1/images/Non_fractured/4-rotated2.jpg
    /kaggle/i

In [21]:
# =========================
# REMOVE EXACT DUPLICATES (OPTIONAL BUT RECOMMENDED)
# =========================

duplicates_to_remove = set(b for (_, b) in duplicate_files)

print("Removing duplicate files:", len(duplicates_to_remove))

clean_image_paths = [
    p for p in all_image_paths if p not in duplicates_to_remove
]

print("Images after duplicate removal:", len(clean_image_paths))

Removing duplicate files: 6557
Images after duplicate removal: 8899


In [22]:
# =========================
# PREPROCESSING (EDGE-AWARE)
# =========================
def preprocess_image(path, img_size=CONFIG["img_size"]):
    try:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None or img.size == 0:
            return None
        if img.shape[0] < 50 or img.shape[1] < 50:
            return None

        img = cv2.resize(img, img_size, interpolation=cv2.INTER_LANCZOS4)
        if np.std(img) < 5:
            return None

        clahe = cv2.createCLAHE(2.0, (8,8))
        img = clahe.apply(img)

        edges = cv2.Sobel(img, cv2.CV_32F, 1, 1)
        edges = cv2.normalize(edges, None, 0, 255, cv2.NORM_MINMAX)

        img = 0.8 * img + 0.2 * edges
        img = img.astype(np.float32) / 255.0
        img = np.expand_dims(img, -1)
        return img
    except:
        return None

In [23]:
# =========================
# LOAD DATA (DUPLICATE-FREE, CORRECT LABELING)
# =========================
X, y = [], []

print("\nLoading images after duplicate removal...")

for path in clean_image_paths:
    parent = os.path.basename(os.path.dirname(path)).lower()

    if parent == "fractured":
        label = 1
    elif parent == "non_fractured":
        label = 0
    else:
        continue

    img = preprocess_image(path)
    if img is not None:
        X.append(img)
        y.append(label)

X = np.array(X)
y = np.array(y)

print("\nDataset loaded (duplicate-free)")
print("Total images:", len(X))
print("Non-fractured:", np.sum(y == 0))
print("Fractured:", np.sum(y == 1))
print("Shape:", X.shape)

Premature end of JPEG file
Premature end of JPEG file



Loading images after duplicate removal...


Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
P


Dataset loaded (duplicate-free)
Total images: 8899
Non-fractured: 5612
Fractured: 3287
Shape: (8899, 224, 224, 1)


In [24]:
# =========================
# TRAIN / TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=CONFIG["test_size"],
    stratify=y,
    random_state=CONFIG["seed"]
)

In [25]:
# =========================
# CLASS WEIGHTS (FN-OPTIMIZED)
# =========================
n0 = np.sum(y_train == 0)
n1 = np.sum(y_train == 1)
total = len(y_train)

class_weight = {
    0: total / (2.0 * n0) * 0.6,
    1: total / (2.0 * n1) * 2.0
}

print("Class weights:", class_weight)

Class weights: {0: 0.4757629761639563, 1: 2.7068441064638784}


In [26]:
# =========================
# DATA AUGMENTATION (SOFTER FOR 224x224)
# =========================
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomZoom(0.08),
    layers.RandomContrast(0.12),
])

In [27]:
# =========================
# MODEL
# =========================
def build_model():
    inputs = layers.Input((224,224,1))
    x = data_augmentation(inputs)

    for filters in [48, 96, 192]:
        x = layers.Conv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Conv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    att = layers.Conv2D(1, 1, activation="sigmoid")(x)
    x = layers.multiply([x, att])

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    return keras.Model(inputs, outputs)

In [28]:
# =========================
# FOCAL LOSS
# =========================
def focal_loss(gamma=2.5, alpha=0.75):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1-1e-7)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1-y_pred)
        return -alpha * tf.pow(1-pt, gamma) * tf.math.log(pt)
    return loss

model = build_model()

model.compile(
    optimizer=keras.optimizers.AdamW(
        learning_rate=CONFIG["learning_rate"],
        weight_decay=1e-4
    ),
    loss=focal_loss(),
    metrics=[
        "accuracy",
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 224, 224,  │          0 │ input_layer_2[0]… │
│ (Sequential)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 224, 224,  │        480 │ sequential_1[0][… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │        192 │ conv2d_7[0][0]    │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_6 (ReLU)      │ (None, 224, 224,  │          0 │ batch_normalizat… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 224, 224,  │     20,784 │ re_lu_6[0][0]     │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │        192 │ conv2d_8[0][0]    │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_7 (ReLU)      │ (None, 224, 224,  │          0 │ batch_normalizat… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 112, 112,  │          0 │ re_lu_7[0][0]     │
│ (MaxPooling2D)      │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 112, 112,  │          0 │ max_pooling2d_3[… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 112, 112,  │     41,568 │ dropout_4[0][0]   │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        384 │ conv2d_9[0][0]    │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_8 (ReLU)      │ (None, 112, 112,  │          0 │ batch_normalizat… │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 112, 112,  │     83,040 │ re_lu_8[0][0]     │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        384 │ conv2d_10[0][0]   │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_9 (ReLU)      │ (None, 112, 112,  │          0 │ batch_normalizat… │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 56, 56,    │          0 │ re_lu_9[0][0]   

 Total params: 696,466 (2.66 MB)

 Trainable params: 695,122 (2.65 MB)

 Non-trainable params: 1,344 (5.25 KB)

In [29]:
# =========================
# CALLBACKS
# =========================
cb = [
    callbacks.EarlyStopping(
        monitor="val_auc",
        patience=35,
        restore_best_weights=True,
        mode="max"
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=12,
        min_lr=1e-7
    )
]

In [30]:
# =========================
# TRAIN
# =========================
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
    class_weight=class_weight,
    callbacks=cb,
    verbose=1
)

Epoch 1/120


E0000 00:00:1767461331.808715      47 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_3_1/dropout_4_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


445/445 ━━━━━━━━━━━━━━━━━━━━ 134s 287ms/step - accuracy: 0.4204 - auc: 0.6768 - loss: 0.0921 - recall: 0.9331 - val_accuracy: 0.3691 - val_auc: 0.5667 - val_loss: 0.2478 - val_recall: 1.0000 - learning_rate: 3.0000e-04
Epoch 2/120
445/445 ━━━━━━━━━━━━━━━━━━━━ 124s 278ms/step - accuracy: 0.4851 - auc: 0.7948 - loss: 0.0784 - recall: 0.9444 - val_accuracy: 0.5034 - val_auc: 0.8510 - val_loss: 0.1022 - val_recall: 0.9696 - learning_rate: 3.0000e-04
Epoch 3/120
445/445 ━━━━━━━━━━━━━━━━━━━━ 124s 278ms/step - accuracy: 0.5135 - auc: 0.8119 - loss: 0.0757 - recall: 0.9327 - val_accuracy: 0.4826 - val_auc: 0.8596 - val_loss: 0.1097 - val_recall: 0.9756 - learning_rate: 3.0000e-04
Epoch 4/120
445/445 ━━━━━━━━━━━━━━━━━━━━ 124s 278ms/step - accuracy: 0.5522 - auc: 0.8295 - loss: 0.0732 - recall: 0.9265 - val_accuracy: 0.4084 - val_auc: 0.8161 - val_loss: 0.1612 - val_recall: 0.9848 - learning_rate: 3.0000e-04
Epoch 5/120
445/445 ━━━━━━━━━━━━━━━━━━━━ 124s 278ms/step - accuracy: 0.5784 - auc: 0.831

In [31]:
# =========================
# THRESHOLD SWEEP
# =========================
y_prob = model.predict(X_test).flatten()

print("\n🔍 Threshold sweep:\n")
results = []

for t in [0.30,0.35,0.40,0.45,0.50,0.55,0.60]:
    y_pred = (y_prob > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    acc = (tp + tn) / (tp + tn + fp + fn)
    recall = tp / (tp + fn)
    spec = tn / (tn + fp)

    results.append((t, acc, recall))
    print(f"t={t:.2f} | Acc={acc*100:.2f}% | Recall={recall:.4f} | FP={fp} | FN={fn}")

56/56 ━━━━━━━━━━━━━━━━━━━━ 9s 125ms/step

🔍 Threshold sweep:

t=0.30 | Acc=74.27% | Recall=0.9756 | FP=442 | FN=16
t=0.35 | Acc=78.37% | Recall=0.9635 | FP=361 | FN=24
t=0.40 | Acc=82.42% | Recall=0.9528 | FP=282 | FN=31
t=0.45 | Acc=86.18% | Recall=0.9467 | FP=211 | FN=35
t=0.50 | Acc=88.31% | Recall=0.9239 | FP=158 | FN=50
t=0.55 | Acc=91.24% | Recall=0.9087 | FP=96 | FN=60
t=0.60 | Acc=92.19% | Recall=0.8798 | FP=60 | FN=79


In [32]:
# =========================
# BEST THRESHOLD (BALANCED)
# =========================
best_t, best_acc = 0, 0
for t, acc, recall in results:
    if acc > best_acc and recall > 0.90:
        best_acc = acc
        best_t = t

print(f"\n🎯 Best threshold: {best_t}")
print(f"Accuracy: {best_acc*100:.2f}%")


🎯 Best threshold: 0.55
Accuracy: 91.24%


In [33]:
# =========================
# FINAL EVALUATION
# =========================
y_final = (y_prob > best_t).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_final).ravel()

print("\nFINAL CONFUSION MATRIX")
print(confusion_matrix(y_test, y_final))

print("\nCLASSIFICATION REPORT")
print(classification_report(y_test, y_final, digits=4))

accuracy = (tp + tn) / (tp + tn + fp + fn)
recall = tp / (tp + fn)
specificity = tn / (tn + fp)

print("\nFINAL METRICS")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Recall (Fracture): {recall:.4f}")
print(f"Specificity: {specificity:.4f}")


FINAL CONFUSION MATRIX
[[1027   96]
 [  60  597]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0     0.9448    0.9145    0.9294      1123
           1     0.8615    0.9087    0.8844       657

    accuracy                         0.9124      1780
   macro avg     0.9031    0.9116    0.9069      1780
weighted avg     0.9140    0.9124    0.9128      1780


FINAL METRICS
Accuracy: 91.24%
Recall (Fracture): 0.9087
Specificity: 0.9145


In [34]:
# =========================
# SAVE MODEL + CONFIG
# =========================
model.save("fracture_model_224_final.keras")

with open("deployment_config.json", "w") as f:
    json.dump({
        "best_threshold": best_t,
        "accuracy": float(accuracy),
        "recall": float(recall),
        "specificity": float(specificity)
    }, f, indent=2)

print("\n✅ Model and deployment config saved")


✅ Model and deployment config saved
